#### Package importing

In [1]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.instantiate();

  Activating project at `~/OneDrive_xwc@zju.edu.cn/optimal-control/my-notes/code/with-notes/lec2`


In [2]:
using LinearAlgebra
using ForwardDiff
using PyPlot
using LaTeXStrings
using JupyterFormatter
enable_autoformat()
PyPlot.rc("text", usetex = true)
PyPlot.rc("font", family = "Times New Roman")
PyPlot.rc("text.latex", preamble = "\\usepackage{{amsmath}}")

#### Pendulum Dynamics

In [3]:
function pendulum_dynamics(x)
    l = 1.0
    g = 9.81

    theta = x[1]
    dot_theta = x[2]

    ddot_theta = -(g / l) * sin(theta)

    return [dot_theta, ddot_theta]
end

pendulum_dynamics (generic function with 1 method)

In [4]:
# initial state
const x0 = [0.1, 0];

### Explicit Method

#### Forward Euler Integrator

In [ ]:
function pendulum_forward_euler(fun, x0, Tf, h)
    
end

In [ ]:
plot_integrator(pendulum_forward_euler)

In [ ]:
# sanity check (stability)
let
    Ad = ForwardDiff.jacobian(x -> forward_euler(pendulum_dynamics, x, 0.1), x0)
    eigvals(Ad)
end

In [ ]:
# plot eigen values of forward euler pendulum
let
    h = collect(0.01:0.01:1)
    eignorm = zeros(length(h))
    for k = 1:length(h)
        eignorm[k] = max(
            norm.(
                eigvals(
                    ForwardDiff.jacobian(
                        x -> forward_euler(pendulum_dynamics, x, h[k]),
                        x0,
                    ),
                ),
            )...,
        )
    end
    plot(h, eignorm)
end

#### RK4 Integrator

In [ ]:
# RK4 implementation
function RK4(dyna, xk, h)
    f1 = dyna(xk)
    f2 = dyna(xk + 0.5 * h * f1)
    f3 = dyna(xk + 0.5 * h * f2)
    f4 = dyna(xk + h * f3)
    return xk + (h / 6.0) * (f1 + 2 * f2 + 2 * f3 + f4)
end

pendulum_RK4 = (dyna, x0, Tf, h) -> pendulum_sim(RK4, dyna, x0, Tf, h)

In [ ]:
plot_integrator(pendulum_RK4)

In [ ]:
# plot eigen values of RK4 pendulum
let
    h = collect(0.01:0.01:1)
    eignorm = zeros(length(h))
    for k = 1:length(h)
        eignorm[k] = max(
            norm.(
                eigvals(ForwardDiff.jacobian(x -> RK4(pendulum_dynamics, x, h[k]), x0)),
            )...,
        )
    end
    plot(h, eignorm)
end

### Implicit Method

#### Backward Euler

In [ ]:
function backward_euler(dyna, xk, h)
    e = 1
    xk1 = zeros(2)
    xk1 .= xk
    while e > 1e-8
        xn = xk + h .* dyna(xk1)
        e = norm(xn - xk1)
        xk1 .= xn
    end
    return xk1
end

In [ ]:
pendulum_backward_euler1 =
    (dyna, x0, Tf, h) -> pendulum_sim(backward_euler, dyna, x0, Tf, h)

In [ ]:
plot_integrator(pendulum_backward_euler1)